In [2]:
%pip install xgboost

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
    --------------------------------------- 1.0/69.5 MB 7.1 MB/s eta 0:00:10
    --------------------------------------- 1.0/69.5 MB 7.1 MB/s eta 0:00:10
    --------------------------------------- 1.0/69.5 MB 7.1 MB/s eta 0:00:10
    --------------------------------------- 1.0/69.5 MB 7.1 MB/s eta 0:00:10
    --------------------------------------- 1.0/69.5 MB 7.1 MB/s eta 0:00:10
   - -------------------------------------- 2.1/69.5 MB 1.6 MB/s eta 0:00:44
   - -------------------------------------- 3.1/69.5 MB 2.0 MB/s eta 0:00:34
   -- ------------------------------------- 3.9/69.5 MB 2.2 MB/s eta 0:00:30
   -- ------------------------------------- 4.5/69.5 MB 2.2 MB/s eta 0:00:30
   -- ------------------------------------- 5.0/69.5 MB 2.3 MB/s eta 0:00:29
   --- ------------------------------------ 5.5/69.5 MB 2.3 MB/s eta 0:00:28
   --

In [3]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

from xgboost import XGBClassifier

In [4]:
df = pd.read_csv(
    r"C:\Users\kapoo\burnsightAI\data\cleaned_burnout.csv"
)

print(df.shape)
df.head()

(1470, 41)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,overall_satisfaction,engagement_score,salary_growth_ratio,promotion_gap,commute_score,BurnoutRisk
0,41,1,2,1102,2,1,2,1,1,1,...,6,4,0,5,2.333333,2.0,665.888889,0,1,1
1,49,0,1,279,1,8,1,1,1,2,...,10,7,1,7,3.000000,2.5,466.363636,1,8,1
2,37,1,2,1373,1,2,2,4,1,4,...,0,0,0,0,3.000000,2.5,261.250000,0,2,0
3,33,0,1,1392,1,3,4,1,1,5,...,8,7,3,0,3.333333,3.0,323.222222,3,3,0
4,27,0,2,591,1,2,1,3,1,7,...,2,2,2,2,2.333333,3.0,495.428571,2,2,1


In [5]:
X = df.drop(columns=["BurnoutRisk"])
y = df["BurnoutRisk"]

print(X.shape)
print(y.value_counts())

(1470, 40)
BurnoutRisk
1    1061
0     392
2      17
Name: count, dtype: int64


In [6]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (1029, 40)
Validation: (220, 40)
Test: (221, 40)


In [7]:
model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="mlogloss"
)

model.fit(X_train, y_train)

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,'mlogloss'


In [8]:
val_preds = model.predict(X_val)

print("VALIDATION REPORT\n")
print(classification_report(y_val, val_preds))

print(
    "Precision:",
    precision_score(
        y_val,
        val_preds,
        average="weighted"
    )
)

print(
    "Recall:",
    recall_score(
        y_val,
        val_preds,
        average="weighted"
    )
)

print(
    "F1:",
    f1_score(
        y_val,
        val_preds,
        average="weighted"
    )
)

print(
    "\nConfusion Matrix\n",
    confusion_matrix(y_val, val_preds)
)

VALIDATION REPORT

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        59
           1       0.99      1.00      0.99       159
           2       0.00      0.00      0.00         2

    accuracy                           0.99       220
   macro avg       0.66      0.67      0.66       220
weighted avg       0.98      0.99      0.99       220

Precision: 0.981931112365895
Recall: 0.990909090909091
F1: 0.9863920454545454

Confusion Matrix
 [[ 59   0   0]
 [  0 159   0]
 [  0   2   0]]


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\ProgramData\anaconda3\Lib

In [9]:
MODEL_DIR = r"C:\Users\kapoo\burnsightAI\models"
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(
    model,
    os.path.join(
        MODEL_DIR,
        "burnout_xgb.pkl"
    )
)

print("Model saved successfully.")

Model saved successfully.


In [10]:
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    VotingClassifier
)
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
import numpy as np

rf = RandomForestClassifier(
    n_estimators=1000,
    max_depth=None,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

et = ExtraTreesClassifier(
    n_estimators=1000,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

gb = GradientBoostingClassifier(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=5,
    random_state=42
)

ensemble = VotingClassifier(
    estimators=[
        ("rf", rf),
        ("et", et),
        ("gb", gb)
    ],
    voting="soft"
)

cv = RepeatedStratifiedKFold(
    n_splits=10,
    n_repeats=20,
    random_state=42
)

scores = cross_val_score(
    ensemble,
    X,
    y,
    scoring="f1_weighted",
    cv=cv,
    n_jobs=-1
)

print(scores.mean())
print(scores.std())

0.9895735352976124
0.007307335123384068


In [11]:
from sklearn.ensemble import IsolationForest

anomaly_model = IsolationForest(
    n_estimators=500,
    contamination=0.1,
    random_state=42
)

anomaly_model.fit(X)

,n_estimators,500
,max_samples,'auto'
,contamination,0.1
,max_features,1.0
,bootstrap,False
,n_jobs,None
,random_state,42
,verbose,0
,warm_start,False


In [12]:
# Workforce Capacity Index (0-100)

df["WCI"] = (
    25 * (df["WorkLifeBalance"] / 4)
    + 25 * (df["JobSatisfaction"] / 4)
    + 20 * (df["EnvironmentSatisfaction"] / 4)
    + 15 * (df["JobInvolvement"] / 4)
    + 15 * (df["RelationshipSatisfaction"] / 4)
)

df["WCI"] = df["WCI"].round(2)

print(df["WCI"].describe())

count    1470.000000
mean       68.328231
std        10.944710
min        28.750000
25%        61.250000
50%        68.750000
75%        76.250000
max       100.000000
Name: WCI, dtype: float64


In [13]:
from sklearn.ensemble import IsolationForest

anomaly_features = [
    "MonthlyIncome",
    "DistanceFromHome",
    "JobSatisfaction",
    "WorkLifeBalance",
    "EnvironmentSatisfaction",
    "JobInvolvement",
    "WCI"
]

iso = IsolationForest(
    n_estimators=500,
    contamination=0.10,
    random_state=42
)

df["Anomaly"] = iso.fit_predict(
    df[anomaly_features]
)

df["AnomalyScore"] = (
    -iso.decision_function(
        df[anomaly_features]
    )
)

print(df["Anomaly"].value_counts())

Anomaly
 1    1323
-1     147
Name: count, dtype: int64


In [14]:
simulation = df.copy()

simulation["Sim_WLB"] = np.minimum(
    simulation["WorkLifeBalance"] + 1,
    4
)

simulation["Sim_JS"] = np.minimum(
    simulation["JobSatisfaction"] + 1,
    4
)

simulation["Sim_ES"] = np.minimum(
    simulation["EnvironmentSatisfaction"] + 1,
    4
)

simulation["Future_WCI"] = (
    25 * (simulation["Sim_WLB"] / 4)
    + 25 * (simulation["Sim_JS"] / 4)
    + 20 * (simulation["Sim_ES"] / 4)
    + 15 * (simulation["JobInvolvement"] / 4)
    + 15 * (simulation["RelationshipSatisfaction"] / 4)
).round(2)

simulation["WCI_Gain"] = (
    simulation["Future_WCI"]
    - simulation["WCI"]
).round(2)

simulation[
    [
        "WCI",
        "Future_WCI",
        "WCI_Gain"
    ]
].head()

,WCI,Future_WCI,WCI_Gain
0,56.25,67.50,11.25
1,68.75,86.25,17.50
2,72.50,85.00,12.50
3,80.00,92.50,12.50
4,62.50,80.00,17.50


In [15]:
import os

MODEL_PATH = r"C:\Users\kapoo\burnsightAI\models\burnout_xgb.pkl"

print(os.path.exists(MODEL_PATH))

True
